# Appendix: Deploying to a Snapdragon Device

This guide walks through packaging your context hub for deployment on a Snapdragon-powered device (Android phone or dev kit). The concepts from lessons 2-7 translate directly to on-device execution.

## Architecture on Device

```
Snapdragon Device
+------------------------------------------+
| App Process                               |
|  +-------------+    +------------------+  |
|  | AI Hub      |    | Qdrant Edge      |  |
|  | (NPU/GPU)   |--->| (EdgeShard)      |  |
|  | Embedding   |    | Local storage    |  |
|  | model       |    |                  |  |
|  +-------------+    +------------------+  |
|         |                    |             |
|         v                    v             |
|  Camera/Sensors        Sync Queue          |
+------------------------------------------+
                               |
                          (when online)
                               v
                        Qdrant Cloud
```

## Step 1: Compile Model for Snapdragon via AI Hub

Use AI Hub to compile your embedding model for the target device. This optimizes the model to run on the Snapdragon NPU.

```python
import qai_hub

# Submit compilation job
compile_job = qai_hub.submit_compile_job(
    model="models/bge-small-en-v1.5.onnx",
    device=qai_hub.Device("Samsung Galaxy S24"),
    input_specs=dict(
        input_ids=(1, 512),
        attention_mask=(1, 512),
    ),
)

# Download the compiled model
target_model = compile_job.get_target_model()
target_model.download("compiled_model.tflite")
```

The compiled model runs directly on the Snapdragon NPU for maximum performance.

## Step 2: Profile On-Device Performance

Before deploying, profile the model to understand latency and resource usage on the target device.

```python
profile_job = qai_hub.submit_profile_job(
    model=target_model,
    device=qai_hub.Device("Samsung Galaxy S24"),
)

profile = profile_job.download_profile()
print(f"Inference time: {profile['inference_time_ms']:.1f} ms")
print(f"Peak memory: {profile['peak_memory_mb']:.1f} MB")
```

## Step 3: Package for Android

Structure your deployment package:

```
context_hub_app/
  models/
    embedding_model.tflite      # Compiled via AI Hub
  qdrant_data/
    edge_shard/                  # Qdrant Edge shard directory
  app.py                         # Main application logic
  requirements.txt               # qdrant-edge-py, etc.
```

Key considerations for mobile deployment:
- **Storage**: EdgeShard directory lives in the app's private storage
- **Models**: Pre-downloaded and bundled with the app
- **Permissions**: Camera access for frame capture
- **Background sync**: Use Android WorkManager for periodic cloud sync

## Step 4: Run on Device

The on-device code is identical to what you wrote in the notebooks:

```python
from qdrant_edge import EdgeShard, EdgeConfig, VectorDataConfig, Distance
from qdrant_edge import Point, UpdateOperation, Query, QueryRequest

# Initialize EdgeShard (same API as in lessons)
config = EdgeConfig(
    vector_data={
        "memory": VectorDataConfig(size=384, distance=Distance.Cosine)
    }
)
shard = EdgeShard("/data/app/qdrant_data/edge_shard", config)

# Capture and store (same as L3)
embedding = run_inference(compiled_model, input_data)
point = Point(id=next_id, vector={"memory": embedding}, payload={...})
shard.update(UpdateOperation.upsert_points([point]))

# Query (same as L4)
results = shard.query(QueryRequest(
    query=Query.Nearest(query_embedding, using="memory"),
    limit=5,
    with_payload=True,
    filter={"must": [{"key": "timestamp", "range": {"gte": cutoff}}]}
))
```

## Step 5: Set Up Cloud Sync

Use the dual-shard pattern from L5:

```python
# Periodic sync (runs when device has connectivity)
def sync_to_cloud():
    while sync_queue:
        batch = sync_queue.pop_batch(size=50)
        cloud_client.upsert(
            collection_name="device_memories",
            points=batch,
        )

def sync_from_cloud():
    manifest = immutable_shard.snapshot_manifest()
    # POST manifest to server for partial snapshot
    # Download and apply partial snapshot
    immutable_shard.update_from_snapshot(snapshot_path)
```

## Performance Targets

Based on the Qdrant Context Hub POC:

| Metric | Target |
|--------|--------|
| Single query latency (P95) | < 20ms |
| Relevant results returned | > 95% |
| Cascade query success (2 devices) | > 95% |
| Vector capacity | 1M+ embeddings |
| Battery impact | < 15% additional drain/day |

Qdrant Edge is designed to meet these targets on Snapdragon chipsets.

## Resources

- [Qdrant Edge documentation](https://qdrant.tech/documentation/edge/)
- [AI Hub](https://aihub.qualcomm.com/)
- [Qdrant Edge Demo (smart glasses)](https://github.com/qdrant/qdrant-edge-demo)
- [Introduction to On-Device AI (Krishna Sridhar)](https://www.deeplearning.ai/short-courses/introduction-to-on-device-ai/)